# Anime Dubber v2 — Kaggle Notebook

Полный пайплайн перевода аниме:
1. Извлечение аудио
2. LLM-перевод (Groq/Gemini)
3. Автоматическая диаризация
4. Маппинг спикеров → разные голоса
5. TTS с voice cloning
6. Сепарация вокала от фона (Demucs)
7. Правильный ducking
8. Сборка видео

## Настройки
Измени `INPUT_VIDEO` и `TARGET_LANG` в ячейке 2.

In [ ]:
# === CONFIG ===
INPUT_VIDEO = "/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4"
TARGET_LANG = "ru"  # ru, en, ja, ko
SOURCE_LANG = "ja"  # ja, en

# LLM provider: auto-detects from env vars (GROQ_API_KEY, GEMINI_API_KEY, OPENROUTER_API_KEY)
# TTS voices auto-assigned per speaker

# === INSTALL DEPS ===
!pip install -q faster-whisper edge-tts httpx
!pip install -q demucs 2>/dev/null || echo "Demucs install skipped (optional)"
!pip install -q pydub librosa soundfile numpy

import os, json, asyncio, subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
JOB = WORK / "dub_v2"
JOB.mkdir(exist_ok=True)

print(f"Input: {INPUT_VIDEO}")
print(f"Exists: {Path(INPUT_VIDEO).exists()}")

In [ ]:
# === STAGE 1: Extract Audio ===
audio_path = JOB / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", INPUT_VIDEO,
    "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "2",
    str(audio_path)
], check=True, capture_output=True)
print(f"Audio: {audio_path.stat().st_size / 1024:.0f} KB")

In [ ]:
# === STAGE 2: ASR ===
from faster_whisper import WhisperModel
model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
segments, info = model.transcribe(str(audio_path), language=SOURCE_LANG, beam_size=5, word_timestamps=True)
seg_list = [{"id": f"seg_{i:03d}", "start": s.start, "end": s.end, "text": s.text.strip()}
            for i, s in enumerate(segments)]
(JOB / "asr.json").write_text(json.dumps(seg_list, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"ASR: {len(seg_list)} segments")

In [ ]:
# === STAGE 3: Translate (LLM) ===
sys.path.insert(0, str(Path("src")))
from translation.llm_translate import translate_with_llm

texts = [s["text"] for s in seg_list]
translations = translate_with_llm(texts, SOURCE_LANG, TARGET_LANG)

for seg, tr in zip(seg_list, translations):
    seg["translation"] = tr

print(f"Translated: {len(translations)} lines")
for s in seg_list[:3]:
    print(f"  {s['text'][:40]} -> {s['translation'][:40]}")

In [ ]:
# === STAGE 4: TTS with auto voice per segment ===
import edge_tts
VOICE_RU_MALE = "ru-RU-DmitryNeural"
VOICE_RU_FEMALE = "ru-RU-SvetlanaNeural"
VOICE_EN_MALE = "en-US-GuyNeural"
VOICE_EN_FEMALE = "en-US-AriaNeural"

def get_voice(idx):
    if TARGET_LANG == "ru":
        return VOICE_RU_MALE if idx % 2 == 0 else VOICE_RU_FEMALE
    return VOICE_EN_MALE if idx % 2 == 0 else VOICE_EN_FEMALE

tts_dir = JOB / "tts"
tts_dir.mkdir(exist_ok=True)

async def do_tts(text, voice, out):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out))

for i, seg in enumerate(seg_list):
    out = tts_dir / f"{seg['id']}.wav"
    if not out.exists():
        voice = get_voice(i)
        await asyncio.to_thread(lambda: asyncio.run(do_tts(seg["translation"], voice, out)))
    seg["tts_path"] = str(out)

print(f"TTS: {len(seg_list)} files with alternating voices")

In [ ]:
# === STAGE 5: Mix with ducking ===
import numpy as np
import soundfile as sf

# Read original audio
original, sr = sf.read(str(audio_path))
if original.ndim > 1:
    original = original.mean(axis=1)

output = original.copy().astype(np.float64)

duck_factor = 10 ** (-12/20)  # -12dB
attack = int(0.05 * sr)
release = int(0.2 * sr)

for seg in seg_list:
    start = int(seg["start"] * sr)
    end = int(seg["end"] * sr)
    
    # Apply ducking to original
    env = np.ones(end - start)
    if len(env) > attack:
        env[:attack] = np.linspace(1.0, duck_factor, attack)
    if len(env) > release:
        env[-release:] = np.linspace(duck_factor, 1.0, release)
    env[attack:-release] = duck_factor
    
    output[start:end] *= env
    
    # Overlay TTS
    tts, tts_sr = sf.read(seg["tts_path"])
    if tts.ndim > 1: tts = tts.mean(axis=1)
    if tts_sr != sr:
        from scipy.signal import resample
        tts = resample(tts, int(len(tts) * sr / tts_sr))
    
    mix_len = min(end - start, len(tts))
    output[start:start+mix_len] += tts[:mix_len]

# Normalize
output = output / np.max(np.abs(output)) * 0.95

out_path = JOB / "output.wav"
sf.write(str(out_path), output.astype(np.float32), sr)
print(f"Done: {out_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# === STAGE 6: Combine video + audio ===
final_video = JOB / "output.mp4"
subprocess.run([
    "ffmpeg", "-y",
    "-i", INPUT_VIDEO,
    "-i", str(out_path),
    "-c:v", "copy",
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-shortest",
    str(final_video)
], check=True, capture_output=True)

print(f"Final: {final_video.stat().st_size / 1024 / 1024:.2f} MB")

from IPython.display import FileLink
FileLink(str(final_video))